
# QML-SleepNet — Stage 04 Quantum Module — AUDITED v3

This notebook is rebuilt after re-auditing the supplied QML-SleepNet pipeline.

## Non-negotiable project priorities enforced here

1. Highest **legitimate** metrics achievable.
2. Exact alignment with the supplied pipeline wherever the pipeline is explicit.
3. No leakage from official `x01–x35` labels.
4. Reuse valid existing artifacts instead of recomputing them.
5. Do not add unrelated architectures, gates, promotion rules, Pareto rules, or old Stage15 methodology.
6. Preserve fold-specific representations so the later Stage-06 hybrid can be evaluated without representation leakage.

---

# Source-locked Stage-04 specification

## Data Encoding Layer
- Angle encoding: `Rx(x_i)`
- **Dense IQP encoding**
- 8 qubits

## Classical Encoder / Classical→Quantum Bridge
- `128 → 64 → 32 → 8`
- BatchNorm + ReLU + Dropout
- dimensionality reduction
- classical→quantum bridge
- **trainable preprocessing**

### Important metric correction
The classical encoder is therefore **not frozen during VQC / Quantum-Transformer training** in this notebook.
Each quantum fold starts from the already-validated fold-specific bridge weights and then jointly fine-tunes
the classical encoder with the quantum branch. This preserves valid artifact reuse while allowing the
trainable preprocessing requested by the pipeline to adapt to quantum geometry.

## Quantum State Preparation
- computational basis state `|0>^8`
- parameterised encoding unitary `U(x)`
- statevector simulation
- Pauli-Z expectation values

## VQC — Proposed QML Core
- StronglyEntanglingLayers
- CNOT entanglers
- Ry/Rz rotation structure (`Rot = Rz-Ry-Rz`)
- **fixed depth L = 4**
- 8 qubits
- **96 trainable quantum parameters = 4 × 8 × 3**

## Quantum Kernel
- fidelity kernel `|<phi(x_i)|phi(x_j)>|²`
- QSVM / classical SVM on top
- uses the same Angle and **true dense-IQP** state preparations

## Quantum Transformer
The guide specifies this block functionally, not by an exact attention equation:
- Q-attention quantum circuits
- superposition-based representation
- quantum self-attention
- hybrid quantum-classical layers
- latent vector 32–64

Therefore the transformer section below is explicitly labelled a **guide-compliant instantiation** rather than
claiming that its internal attention equation was specified by the source. Both guide encodings are evaluated
instead of prematurely selecting one from standalone VQC performance.

## Deliberately excluded
- amplitude encoding
- VQC depths 5/6
- STGCN
- old Stage15/Pareto/gate machinery
- train caps
- official-test-label tuning
- threshold tuning
- final-hybrid use of the raw classical bridge as an extra feature

The Stage-06 proposed hybrid calls for **8-D quantum features**; therefore the 8-D VQC measurements are the
primary Stage-06 QML payload. Quantum-Transformer 64-D vectors are retained as a separate guide branch/comparator.


In [ ]:

# Cell 1 — environment, validated upstream artifacts, and paths
!pip -q install "pennylane>=0.40" scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc, json, math, os, random, time, hashlib, warnings
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
    log_loss
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml

warnings.filterwarnings("ignore")

SEED = 42
N_QUBITS = 8
DEPTH = 4
VQC_QPARAMS = DEPTH * N_QUBITS * 3
WIRES = list(range(N_QUBITS))
DENSE_PAIRS = [(i,j) for i in range(N_QUBITS) for j in range(i+1,N_QUBITS)]

# default.qubit is a CPU statevector simulator.
QML_DEVICE = torch.device("cpu")

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
BRIDGE = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_bridge128to8_v1_3_fullgrid"
WINFOLDS = BRIDGE / "winning_folds"
FINAL_BRIDGE = BRIDGE / "final_fit"

OUT = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_quantum_module_v3_audited"
VQC_DIR = OUT / "vqc"
KERNEL_DIR = OUT / "quantum_kernel"
QT_DIR = OUT / "quantum_transformer"
FOLD_STAGE06_DIR = OUT / "stage06_fold_quantum_features"
FINAL_DIR = OUT / "final_fit"
for p in (OUT,VQC_DIR,KERNEL_DIR,QT_DIR,FOLD_STAGE06_DIR,FINAL_DIR):
    p.mkdir(parents=True, exist_ok=True)

QREADY = FINAL_BRIDGE / "quantum_ready_8d.npz"
FINAL_BRIDGE_MODEL = FINAL_BRIDGE / "bridge128to8_model.pt"
if not QREADY.is_file():
    raise FileNotFoundError(QREADY)
if not FINAL_BRIDGE_MODEL.is_file():
    raise FileNotFoundError(FINAL_BRIDGE_MODEL)

qf = np.load(QREADY, allow_pickle=False)

FINAL_Z128_LEARN = np.asarray(qf["Z128_learn"], np.float32)
FINAL_Z128_TEST = np.asarray(qf["Z128_test"], np.float32)
FINAL_Z8_LEARN = np.asarray(qf["Z8_learn"], np.float32)
FINAL_Z8_TEST = np.asarray(qf["Z8_test"], np.float32)
Y_LEARN = np.asarray(qf["y_learn"], np.int8)
UID_LEARN = np.asarray(qf["learn_uids"]).astype(str)
UID_TEST = np.asarray(qf["test_uids"]).astype(str)

FOLDS = {}
for fold in range(5):
    data_path = WINFOLDS / f"bridge_fold{fold}_data.npz"
    model_path = WINFOLDS / f"bridge_fold{fold}.pt"
    if not data_path.is_file():
        raise FileNotFoundError(data_path)
    if not model_path.is_file():
        raise FileNotFoundError(model_path)

    d = np.load(data_path, allow_pickle=False)
    FOLDS[fold] = {
        "tr": np.asarray(d["train_rows"], np.int64),
        "va": np.asarray(d["val_rows"], np.int64),
        "ytr": np.asarray(d["y_train"], np.int8),
        "yva": np.asarray(d["y_val"], np.int8),
        "Z128tr": np.asarray(d["Z128_train"], np.float32),
        "Z128va": np.asarray(d["Z128_val"], np.float32),
        "Z8tr": np.asarray(d["Z8_train"], np.float32),
        "Z8va": np.asarray(d["Z8_val"], np.float32),
        "bridge_model_path": model_path,
    }

assert FINAL_Z128_LEARN.shape[1] == 128
assert FINAL_Z128_TEST.shape[1] == 128
assert FINAL_Z8_LEARN.shape[1] == 8
assert FINAL_Z8_TEST.shape[1] == 8
assert len(DENSE_PAIRS) == 28
assert VQC_QPARAMS == 96

print("PennyLane:", qml.__version__)
print("Torch:", torch.__version__)
print("Quantum execution device:", QML_DEVICE)
print("Learn Z128:", FINAL_Z128_LEARN.shape)
print("Official test Z128:", FINAL_Z128_TEST.shape)
print("Fixed VQC depth:", DEPTH)
print("VQC quantum parameters:", VQC_QPARAMS)
print("Dense IQP ZZ pairs:", len(DENSE_PAIRS))
print("Duplicate-safe fold artifacts loaded:", len(FOLDS))


In [ ]:

# Cell 2 — hard source-audit guard

SOURCE_LOCK = {
    "angle_encoding": "Rx(x_i)",
    "iqp_encoding": "dense",
    "qubits": 8,
    "classical_encoder": "128->64->32->8",
    "classical_encoder_trainable": True,
    "vqc_ansatz": "StronglyEntanglingLayers",
    "entangler": "CNOT",
    "rotation_structure": "Rz-Ry-Rz (contains Ry/Rz)",
    "vqc_depth": 4,
    "vqc_quantum_parameter_count": 96,
    "measurement": "Pauli-Z expectations",
    "simulator": "statevector",
    "quantum_kernel": "fidelity",
    "kernel_classifier": "classical SVM/QSVM",
    "quantum_transformer_output_dim": 64,
    "official_test_labels_used_for_training_or_selection": False,
}

print(json.dumps(SOURCE_LOCK, indent=2))

assert SOURCE_LOCK["angle_encoding"] == "Rx(x_i)"
assert SOURCE_LOCK["iqp_encoding"] == "dense"
assert SOURCE_LOCK["qubits"] == 8
assert SOURCE_LOCK["vqc_depth"] == 4
assert SOURCE_LOCK["vqc_quantum_parameter_count"] == 96
assert SOURCE_LOCK["classical_encoder_trainable"] is True


In [ ]:

# Cell 3 — deterministic metrics and summary helpers

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def metrics_from_score(y, score, pred=None):
    score=np.asarray(score,float)
    if pred is None:
        pred=(score>=0.5).astype(np.int8)
    pred=np.asarray(pred,np.int8)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "precision":float(precision_score(y,pred,zero_division=0)),
        "recall":float(recall_score(y,pred,zero_division=0)),
        "f1":float(f1_score(y,pred,zero_division=0)),
        "auroc":float(roc_auc_score(y,score)),
        "auprc":float(average_precision_score(y,score)),
        "mcc":float(matthews_corrcoef(y,pred)),
    }

def rank_summary(df, group_cols):
    g=df.groupby(group_cols,as_index=False)
    mean=g.agg(
        mean_accuracy=("accuracy","mean"),
        mean_balanced_accuracy=("balanced_accuracy","mean"),
        mean_precision=("precision","mean"),
        mean_recall=("recall","mean"),
        mean_f1=("f1","mean"),
        mean_auroc=("auroc","mean"),
        mean_auprc=("auprc","mean"),
        mean_mcc=("mcc","mean"),
        std_accuracy=("accuracy","std"),
        std_f1=("f1","std"),
        std_auroc=("auroc","std"),
    )
    return mean.sort_values(
        ["mean_accuracy","mean_auroc","mean_f1"],
        ascending=False
    ).reset_index(drop=True)

def stack_qnode_output(out):
    if isinstance(out,(tuple,list)):
        q=torch.stack(list(out),dim=-1)
    else:
        q=out
    if q.ndim==1:
        q=q.unsqueeze(0)
    return q

print("Helpers ready.")



# A. Classical Encoder + Exact VQC, trained end-to-end

This corrects the most important performance risk in the previous notebook.

The supplied Stage-04 diagram calls the `128→64→32→8` MLP **trainable preprocessing**. Therefore:

- every fold loads the already-validated fold-specific S192/dropout-0.5 bridge weights;
- the classical head is discarded;
- the encoder is jointly fine-tuned with the VQC;
- validation data never updates the model;
- final Stage-06 fold artifacts include **both training and validation quantum features produced by the same fold model**, preventing downstream representation leakage.

The source fixes VQC depth at `L=4`; no deeper circuits are tried.


In [ ]:

# Cell 4 — trainable 128→64→32→8 encoder, initialized from the winning bridge

ENCODER_DROPOUT = 0.5  # winner already established by the full guide-permitted bridge screen

class TrainableEncoder128to8(nn.Module):
    def __init__(self,dropout=ENCODER_DROPOUT):
        super().__init__()
        self.fc1=nn.Linear(128,64)
        self.bn1=nn.BatchNorm1d(64)
        self.fc2=nn.Linear(64,32)
        self.bn2=nn.BatchNorm1d(32)
        self.fc3=nn.Linear(32,8)
        self.drop=nn.Dropout(float(dropout))

    def forward(self,x):
        x=self.drop(F.relu(self.bn1(self.fc1(x))))
        x=self.drop(F.relu(self.bn2(self.fc2(x))))
        return self.fc3(x)

def load_pretrained_encoder(model_path):
    ck=torch.load(model_path,map_location="cpu",weights_only=False)
    sd=ck["state_dict"]
    allowed=("fc1.","bn1.","fc2.","bn2.","fc3.")
    enc_sd={k:v for k,v in sd.items() if k.startswith(allowed)}

    enc=TrainableEncoder128to8(ENCODER_DROPOUT)
    missing,unexpected=enc.load_state_dict(enc_sd,strict=False)

    # Only dropout has no state; every parameter/buffer in the encoder must load.
    if missing or unexpected:
        raise RuntimeError(
            f"Bridge→encoder state mismatch. missing={missing}, unexpected={unexpected}"
        )
    return enc

# Smoke-load every fold encoder and final encoder now, before expensive quantum work.
for fold in range(5):
    e=load_pretrained_encoder(FOLDS[fold]["bridge_model_path"])
    assert e.fc1.weight.shape==(64,128)
    assert e.fc3.weight.shape==(8,32)
    del e

e=load_pretrained_encoder(FINAL_BRIDGE_MODEL)
del e
print("All 5 fold encoders + final encoder load: PASS")


In [ ]:

# Cell 5 — exact differentiable Angle Rx and TRUE DENSE-IQP encodings

ENCODINGS=["angle_rx","iqp_dense"]

def prepare_zero_basis():
    qml.BasisState(np.zeros(N_QUBITS,dtype=np.int8),wires=WIRES)

def apply_angle_rx(x):
    prepare_zero_basis()
    # Manual form guarantees exactly Rx(x_i).
    for i in range(N_QUBITS):
        qml.RX(x[...,i],wires=i)

def apply_dense_iqp(x):
    prepare_zero_basis()

    # Explicit dense IQP:
    # H on every qubit, RZ(x_i), then ZZ/MultiRZ(x_i*x_j) for ALL 28 pairs.
    # This avoids the library template's default neighbour-only pattern and
    # keeps gradients flowing into the trainable classical encoder.
    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)

    for i in range(N_QUBITS):
        qml.RZ(x[...,i],wires=i)

    for i,j in DENSE_PAIRS:
        qml.MultiRZ(x[...,i]*x[...,j],wires=[i,j])

def apply_encoding(x,encoding):
    if encoding=="angle_rx":
        apply_angle_rx(x)
    elif encoding=="iqp_dense":
        apply_dense_iqp(x)
    else:
        raise ValueError(encoding)

print("Dense IQP pair count:",len(DENSE_PAIRS))
assert len(DENSE_PAIRS)==28


In [ ]:

# Cell 6 — joint classical→quantum VQC model

VQC_EPOCHS=60
VQC_BATCH=256

def make_qdevice():
    return qml.device("default.qubit",wires=N_QUBITS)

class JointGuideVQC(nn.Module):
    def __init__(self,encoding,pretrained_encoder):
        super().__init__()
        self.encoding=str(encoding)
        self.encoder=pretrained_encoder

        # Exactly 4×8×3 = 96 trainable quantum parameters.
        self.qweights=nn.Parameter(
            0.05*torch.randn(DEPTH,N_QUBITS,3,dtype=torch.float32)
        )
        self.head=nn.Linear(N_QUBITS,2)

        dev=make_qdevice()
        enc=self.encoding

        @qml.qnode(dev,interface="torch",diff_method="backprop")
        def circuit(x8,qweights):
            apply_encoding(x8,enc)
            qml.StronglyEntanglingLayers(
                qweights,wires=WIRES,imprimitive=qml.CNOT
            )
            return tuple(qml.expval(qml.PauliZ(w)) for w in WIRES)

        self.circuit=circuit

    def quantum_features(self,z128):
        x8=self.encoder(z128)
        out=self.circuit(x8,self.qweights)
        q=stack_qnode_output(out)
        q=q.to(dtype=self.head.weight.dtype,device=self.head.weight.device)
        return q,x8

    def forward(self,z128):
        q,x8=self.quantum_features(z128)
        return self.head(q),q,x8

def train_joint_vqc(
    Ztr,ytr,Zv,yv,encoding,encoder_model_path,seed,
    epochs=VQC_EPOCHS,batch_size=VQC_BATCH,return_train=False
):
    set_seed(seed)
    encoder=load_pretrained_encoder(encoder_model_path)
    model=JointGuideVQC(encoding,encoder).to(QML_DEVICE)

    assert model.qweights.numel()==96

    ds=TensorDataset(
        torch.tensor(Ztr,dtype=torch.float32),
        torch.tensor(ytr,dtype=torch.long)
    )
    gen=torch.Generator().manual_seed(seed)
    dl=DataLoader(ds,batch_size=batch_size,shuffle=True,generator=gen,num_workers=0)

    opt=torch.optim.AdamW(
        model.parameters(),lr=1e-4,weight_decay=1e-4,
        betas=(0.9,0.999),amsgrad=True,eps=1e-8
    )

    warmup=10
    def lr_lambda(ep):
        if ep<warmup:
            return (ep+1)/warmup
        progress=(ep-warmup)/max(epochs-warmup-1,1)
        min_ratio=1e-6/1e-4
        return min_ratio+(1-min_ratio)*0.5*(1+math.cos(math.pi*progress))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_lambda=lr_lambda)

    for ep in range(epochs):
        model.train()
        running=0.0

        for xb,yb in dl:
            xb=xb.to(QML_DEVICE); yb=yb.to(QML_DEVICE)

            logits,_,_=model(xb)
            loss=F.cross_entropy(logits,yb,label_smoothing=0.1)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()

            # Source: VQC parameter clipping. Clip quantum parameters only.
            with torch.no_grad():
                model.qweights.clamp_(-math.pi,math.pi)

            running += float(loss.detach())*len(xb)

        sched.step()

        if (ep+1)%10==0:
            print(
                f"  epoch {ep+1:02d}/{epochs} "
                f"loss={running/len(ds):.5f} "
                f"lr={opt.param_groups[0]['lr']:.2e}"
            )

    @torch.no_grad()
    def infer(Z):
        model.eval()
        probs=[]; qfeats=[]; x8s=[]
        for st in range(0,len(Z),512):
            xb=torch.tensor(
                Z[st:st+512],dtype=torch.float32,device=QML_DEVICE
            )
            logits,q,x8=model(xb)
            probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
            qfeats.append(q.cpu().numpy())
            x8s.append(x8.cpu().numpy())

        return (
            np.concatenate(probs),
            np.concatenate(qfeats).astype(np.float32),
            np.concatenate(x8s).astype(np.float32)
        )

    pv,Qv,X8v=infer(Zv)
    m=metrics_from_score(yv,pv)

    if return_train:
        ptr,Qtr,X8tr=infer(Ztr)
        return model,m,pv,Qv,X8v,ptr,Qtr,X8tr
    return model,m,pv,Qv,X8v


In [ ]:

# Cell 7 — mandatory device/dtype/gradient smoke tests for BOTH encodings

for enc in ENCODINGS:
    set_seed(123)
    encoder=load_pretrained_encoder(FOLDS[0]["bridge_model_path"])
    model=JointGuideVQC(enc,encoder).to(QML_DEVICE)
    model.train()

    xb=torch.tensor(
        FOLDS[0]["Z128tr"][:4],dtype=torch.float32,device=QML_DEVICE
    )
    yb=torch.tensor(FOLDS[0]["ytr"][:4],dtype=torch.long,device=QML_DEVICE)

    logits,qfeat,x8=model(xb)
    loss=F.cross_entropy(logits,yb)
    loss.backward()

    encoder_grad=float(model.encoder.fc1.weight.grad.abs().sum())
    q_grad=float(model.qweights.grad.abs().sum())

    print(enc)
    print("  x8    :",x8.dtype,x8.device,tuple(x8.shape))
    print("  qfeat :",qfeat.dtype,qfeat.device,tuple(qfeat.shape))
    print("  logits:",logits.dtype,logits.device,tuple(logits.shape))
    print("  encoder grad sum:",encoder_grad)
    print("  quantum grad sum:",q_grad)

    assert tuple(x8.shape)==(4,8)
    assert tuple(qfeat.shape)==(4,8)
    assert tuple(logits.shape)==(4,2)
    assert torch.isfinite(x8).all()
    assert torch.isfinite(qfeat).all()
    assert torch.isfinite(logits).all()
    assert encoder_grad>0.0, f"{enc}: encoder is not trainable end-to-end"
    assert q_grad>0.0, f"{enc}: quantum parameters have no gradient"
    assert model.qweights.numel()==96

    del model,encoder,xb,yb,logits,qfeat,x8,loss
    gc.collect()

print("ANGLE + TRUE DENSE-IQP END-TO-END GRADIENT SMOKE TESTS: PASS")


In [ ]:

# Cell 8 — 2 source-listed encodings × fixed depth-4 × 5 folds
# New output root intentionally prevents reuse of older frozen-encoder VQC results.

VQC_ROWS=[]

for enc in ENCODINGS:
    for fold in range(5):
        cp=VQC_DIR/f"{enc}_fold{fold}.npz"

        if cp.is_file():
            q=np.load(cp,allow_pickle=False)
            m=json.loads(str(q["metrics_json"].item()))
            VQC_ROWS.append({"encoding":enc,"fold":fold,**m})
            print(
                "RESUME",enc,"fold",fold,
                "acc",round(m["accuracy"],4),
                "auc",round(m["auroc"],4)
            )
            continue

        d=FOLDS[fold]
        print(f"\nJOINT VQC {enc} depth=4 fold={fold}")
        t0=time.time()

        model,m,pv,Qv,X8v,ptr,Qtr,X8tr=train_joint_vqc(
            d["Z128tr"],d["ytr"],
            d["Z128va"],d["yva"],
            enc,d["bridge_model_path"],
            seed=1000+fold,
            return_train=True
        )
        sec=time.time()-t0

        np.savez_compressed(
            cp,
            train_rows=d["tr"],
            val_rows=d["va"],
            y_train=d["ytr"],
            y_val=d["yva"],
            train_probability=ptr.astype(np.float64),
            val_probability=pv.astype(np.float64),
            q8_train=Qtr.astype(np.float32),
            q8_val=Qv.astype(np.float32),
            encoder8_train=X8tr.astype(np.float32),
            encoder8_val=X8v.astype(np.float32),
            metrics_json=np.asarray(json.dumps(m)),
            seconds=np.asarray(sec,np.float64),
        )

        torch.save(
            {
                "state_dict":model.state_dict(),
                "encoding":enc,
                "depth":4,
                "quantum_parameter_count":96,
                "joint_encoder":True,
            },
            VQC_DIR/f"{enc}_fold{fold}.pt"
        )

        VQC_ROWS.append({"encoding":enc,"fold":fold,**m})
        print("  metrics:",m,"seconds:",round(sec,1))

        del model
        gc.collect()

vqc_df=pd.DataFrame(VQC_ROWS)
vqc_df.to_csv(VQC_DIR/"vqc_fold_metrics.csv",index=False)

vqc_summary=rank_summary(vqc_df,["encoding"])
vqc_summary.to_csv(VQC_DIR/"vqc_summary.csv",index=False)

print("\nJOINT EXACT-GUIDE VQC SUMMARY")
display(vqc_summary)

STANDALONE_VQC_LEADER=str(vqc_summary.iloc[0]["encoding"])
print("Standalone VQC leader:",STANDALONE_VQC_LEADER)
print("Both 8-D quantum encodings remain eligible for Stage-06 hybrid evaluation.")


In [ ]:

# Cell 9 — construct OOF 8-D quantum features for both encodings

VQC_OOF={}

for enc in ENCODINGS:
    oof_prob=np.full(len(Y_LEARN),np.nan,dtype=np.float64)
    oof_q=np.full((len(Y_LEARN),8),np.nan,dtype=np.float32)

    for fold in range(5):
        q=np.load(VQC_DIR/f"{enc}_fold{fold}.npz",allow_pickle=False)
        va=np.asarray(q["val_rows"],np.int64)
        oof_prob[va]=np.asarray(q["val_probability"],np.float64)
        oof_q[va]=np.asarray(q["q8_val"],np.float32)

    assert np.isfinite(oof_prob).all()
    assert np.isfinite(oof_q).all()

    met=metrics_from_score(Y_LEARN,oof_prob)
    VQC_OOF[enc]={"probability":oof_prob,"q8":oof_q,"metrics":met}

    np.savez_compressed(
        VQC_DIR/f"{enc}_oof.npz",
        uids=UID_LEARN.astype("U128"),
        y=Y_LEARN,
        probability=oof_prob,
        q8=oof_q,
    )

    print(enc,json.dumps(met,indent=2))



# B. Exact fidelity Quantum Kernel / QSVM

The quantum-kernel branch is not differentiable through a classical SVM, so it uses the already-trained
fold-specific 8-D classical bridge output (`Z8tr/Z8va`) as its Stage-04 preprocessing.

## Angle fidelity
For the product `Rx` state the fidelity is evaluated analytically, exactly:

\[
K(x,y)=\prod_i \cos^2((x_i-y_i)/2).
\]

## Dense-IQP fidelity
The state is generated with the same **manual all-pairs dense IQP** used above, then

\[
K(x,y)=|\langle\phi(x)|\phi(y)\rangle|^2.
\]

The diagram does not fix SVM `C`, so `C ∈ {0.01,0.1,1,10,100}` is treated as ordinary classifier
hyperparameter tuning on the five learning folds. No train cap is used.


In [ ]:

# Cell 10 — fidelity kernel helpers

C_GRID=[0.01,0.1,1.0,10.0,100.0]

def angle_rx_fidelity_kernel(A,B,block=128):
    A=np.asarray(A,np.float64)
    B=np.asarray(B,np.float64)
    K=np.empty((len(A),len(B)),dtype=np.float32)

    for st in range(0,len(A),block):
        aa=A[st:st+block]
        diff=aa[:,None,:]-B[None,:,:]
        K[st:st+len(aa)] = np.prod(
            np.square(np.cos(0.5*diff)),axis=2
        ).astype(np.float32)
    return K

_state_dev=qml.device("default.qubit",wires=N_QUBITS)

@qml.qnode(_state_dev,interface="numpy")
def dense_iqp_state(x):
    prepare_zero_basis()

    for i in range(N_QUBITS):
        qml.Hadamard(wires=i)
    for i in range(N_QUBITS):
        qml.RZ(x[i],wires=i)
    for i,j in DENSE_PAIRS:
        qml.MultiRZ(x[i]*x[j],wires=[i,j])

    return qml.state()

def dense_iqp_states(X):
    X=np.asarray(X,np.float64)
    out=np.empty((len(X),2**N_QUBITS),dtype=np.complex64)

    for i,x in enumerate(X):
        out[i]=np.asarray(dense_iqp_state(x),np.complex64)
        if (i+1)%2000==0:
            print("    dense-IQP states:",i+1,"/",len(X))
    return out

def fidelity_from_states(A,B,block=256):
    A=np.asarray(A,np.complex64)
    B=np.asarray(B,np.complex64)

    K=np.empty((len(A),len(B)),dtype=np.float32)
    BH=B.conj().T

    for st in range(0,len(A),block):
        aa=A[st:st+block]
        ov=aa@BH
        K[st:st+len(aa)] = np.square(np.abs(ov)).astype(np.float32)

    return K

print("Kernel helpers ready.")


In [ ]:

# Cell 11 — 5-fold fidelity-kernel/QSVM screen, resume-safe

KERNEL_ROWS=[]

for enc in ENCODINGS:
    for fold in range(5):
        result_csv=KERNEL_DIR/f"{enc}_fold{fold}_metrics.csv"

        if result_csv.is_file():
            old=pd.read_csv(result_csv)
            KERNEL_ROWS.extend(old.to_dict(orient="records"))
            print("RESUME kernel",enc,"fold",fold)
            continue

        d=FOLDS[fold]
        print(f"\nQUANTUM KERNEL {enc} fold={fold}")
        t0=time.time()

        if enc=="angle_rx":
            Ktr=angle_rx_fidelity_kernel(d["Z8tr"],d["Z8tr"])
            Kva=angle_rx_fidelity_kernel(d["Z8va"],d["Z8tr"])
        else:
            print("  building dense-IQP train states...")
            Str=dense_iqp_states(d["Z8tr"])
            print("  building dense-IQP validation states...")
            Sva=dense_iqp_states(d["Z8va"])
            print("  building fidelity matrices...")
            Ktr=fidelity_from_states(Str,Str)
            Kva=fidelity_from_states(Sva,Str)
            del Str,Sva
            gc.collect()

        fold_rows=[]
        for C in C_GRID:
            clf=SVC(kernel="precomputed",C=float(C),cache_size=4000)
            clf.fit(Ktr,d["ytr"])

            score=clf.decision_function(Kva)
            pred=clf.predict(Kva)
            m=metrics_from_score(d["yva"],score,pred)

            row={"encoding":enc,"C":float(C),"fold":fold,**m}
            fold_rows.append(row)
            KERNEL_ROWS.append(row)

            print(
                f"  C={C:g}: acc={m['accuracy']:.4f} "
                f"auc={m['auroc']:.4f} f1={m['f1']:.4f}"
            )

        pd.DataFrame(fold_rows).to_csv(result_csv,index=False)

        print("  fold kernel seconds:",round(time.time()-t0,1))
        del Ktr,Kva
        gc.collect()

kernel_df=pd.DataFrame(KERNEL_ROWS)
kernel_df.to_csv(KERNEL_DIR/"quantum_kernel_fold_metrics.csv",index=False)

kernel_summary=rank_summary(kernel_df,["encoding","C"])
kernel_summary.to_csv(KERNEL_DIR/"quantum_kernel_summary.csv",index=False)

print("\nFIDELITY KERNEL / QSVM SUMMARY")
display(kernel_summary)

KERNEL_LEADER_ENCODING=str(kernel_summary.iloc[0]["encoding"])
KERNEL_LEADER_C=float(kernel_summary.iloc[0]["C"])
print("Kernel leader:",KERNEL_LEADER_ENCODING,"C",KERNEL_LEADER_C)



# C. Quantum Transformer — guide-compliant instantiation

The pipeline names the required building blocks but does not give the exact QSA equation.
To avoid silently pretending otherwise, this is a **guide-compliant instantiation**:

- same trainable `128→64→32→8` preprocessing,
- quantum Q / K / V circuits,
- fixed source VQC geometry (`L=4`, CNOT, StronglyEntanglingLayers),
- self-attention across the eight measured quantum coordinates,
- hybrid projection to **64-D**,
- classical head only for branch evaluation.

**Both Angle and dense-IQP encodings are evaluated independently.**
No encoding is chosen merely because it won standalone VQC.


In [ ]:

# Cell 12 — joint-encoder Quantum Transformer

QT_ENCODINGS=ENCODINGS.copy()
QT_LATENT=64
QT_EPOCHS=40
QT_BATCH=192

class JointGuideQuantumTransformer(nn.Module):
    def __init__(self,encoding,pretrained_encoder,latent_dim=64,dropout=0.5):
        super().__init__()
        self.encoding=str(encoding)
        self.encoder=pretrained_encoder
        self.latent_dim=int(latent_dim)

        self.wq=nn.Parameter(0.05*torch.randn(DEPTH,N_QUBITS,3))
        self.wk=nn.Parameter(0.05*torch.randn(DEPTH,N_QUBITS,3))
        self.wv=nn.Parameter(0.05*torch.randn(DEPTH,N_QUBITS,3))

        dev=make_qdevice()
        enc=self.encoding

        @qml.qnode(dev,interface="torch",diff_method="backprop")
        def circuit(x8,weights):
            apply_encoding(x8,enc)
            qml.StronglyEntanglingLayers(
                weights,wires=WIRES,imprimitive=qml.CNOT
            )
            return tuple(qml.expval(qml.PauliZ(w)) for w in WIRES)

        self.circuit=circuit

        self.hybrid=nn.Sequential(
            nn.Linear(N_QUBITS,latent_dim),
            nn.ReLU(),
            nn.Dropout(float(dropout)),
        )
        self.head=nn.Linear(latent_dim,2)

    def qproj(self,x8,w):
        q=stack_qnode_output(self.circuit(x8,w))
        return q.to(
            dtype=self.hybrid[0].weight.dtype,
            device=self.hybrid[0].weight.device
        )

    def latent(self,z128):
        x8=self.encoder(z128)
        q=self.qproj(x8,self.wq)
        k=self.qproj(x8,self.wk)
        v=self.qproj(x8,self.wv)

        # Guide-compliant QSA instantiation across 8 measured coordinates.
        attn_logits=q.unsqueeze(2)*k.unsqueeze(1)/math.sqrt(N_QUBITS)
        attn=torch.softmax(attn_logits,dim=-1)
        attended=torch.bmm(attn,v.unsqueeze(-1)).squeeze(-1)

        return self.hybrid(attended),x8

    def forward(self,z128):
        z64,x8=self.latent(z128)
        return self.head(z64),z64,x8

def train_joint_qt(
    Ztr,ytr,Zv,yv,encoding,encoder_model_path,seed,
    epochs=QT_EPOCHS,batch_size=QT_BATCH,return_train=False
):
    set_seed(seed)
    encoder=load_pretrained_encoder(encoder_model_path)
    model=JointGuideQuantumTransformer(
        encoding,encoder,latent_dim=64,dropout=0.5
    ).to(QML_DEVICE)

    ds=TensorDataset(
        torch.tensor(Ztr,dtype=torch.float32),
        torch.tensor(ytr,dtype=torch.long)
    )
    gen=torch.Generator().manual_seed(seed)
    dl=DataLoader(ds,batch_size=batch_size,shuffle=True,generator=gen,num_workers=0)

    opt=torch.optim.AdamW(
        model.parameters(),lr=1e-4,weight_decay=1e-4,
        betas=(0.9,0.999),amsgrad=True,eps=1e-8
    )

    warmup=10
    def lr_lambda(ep):
        if ep<warmup:
            return (ep+1)/warmup
        progress=(ep-warmup)/max(epochs-warmup-1,1)
        min_ratio=1e-6/1e-4
        return min_ratio+(1-min_ratio)*0.5*(1+math.cos(math.pi*progress))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_lambda=lr_lambda)

    for ep in range(epochs):
        model.train()
        running=0.0

        for xb,yb in dl:
            xb=xb.to(QML_DEVICE); yb=yb.to(QML_DEVICE)
            logits,_,_=model(xb)
            loss=F.cross_entropy(logits,yb,label_smoothing=0.1)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()

            with torch.no_grad():
                for w in (model.wq,model.wk,model.wv):
                    w.clamp_(-math.pi,math.pi)

            running += float(loss.detach())*len(xb)

        sched.step()

        if (ep+1)%10==0:
            print(
                f"  epoch {ep+1:02d}/{epochs} "
                f"loss={running/len(ds):.5f} "
                f"lr={opt.param_groups[0]['lr']:.2e}"
            )

    @torch.no_grad()
    def infer(Z):
        model.eval()
        probs=[]; lat=[]; x8s=[]
        for st in range(0,len(Z),384):
            xb=torch.tensor(
                Z[st:st+384],dtype=torch.float32,device=QML_DEVICE
            )
            logits,z64,x8=model(xb)
            probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
            lat.append(z64.cpu().numpy())
            x8s.append(x8.cpu().numpy())

        return (
            np.concatenate(probs),
            np.concatenate(lat).astype(np.float32),
            np.concatenate(x8s).astype(np.float32)
        )

    pv,Lv,X8v=infer(Zv)
    m=metrics_from_score(yv,pv)

    if return_train:
        ptr,Ltr,X8tr=infer(Ztr)
        return model,m,pv,Lv,X8v,ptr,Ltr,X8tr
    return model,m,pv,Lv,X8v


In [ ]:

# Cell 13 — Quantum Transformer smoke tests for both encodings

for enc in QT_ENCODINGS:
    set_seed(321)
    encoder=load_pretrained_encoder(FOLDS[0]["bridge_model_path"])
    model=JointGuideQuantumTransformer(enc,encoder,64,0.5).to(QML_DEVICE)
    model.train()

    xb=torch.tensor(
        FOLDS[0]["Z128tr"][:4],dtype=torch.float32,device=QML_DEVICE
    )
    yb=torch.tensor(FOLDS[0]["ytr"][:4],dtype=torch.long,device=QML_DEVICE)

    logits,z64,x8=model(xb)
    loss=F.cross_entropy(logits,yb)
    loss.backward()

    enc_grad=float(model.encoder.fc1.weight.grad.abs().sum())
    q_grad=float(
        model.wq.grad.abs().sum()+
        model.wk.grad.abs().sum()+
        model.wv.grad.abs().sum()
    )

    print(enc,tuple(z64.shape),"encoder_grad",enc_grad,"q_grad",q_grad)

    assert tuple(z64.shape)==(4,64)
    assert tuple(logits.shape)==(4,2)
    assert enc_grad>0.0
    assert q_grad>0.0
    assert torch.isfinite(z64).all()

    del model,encoder,xb,yb,logits,z64,x8,loss
    gc.collect()

print("QUANTUM TRANSFORMER END-TO-END SMOKE TESTS: PASS")


In [ ]:

# Cell 14 — both Quantum-Transformer encodings × 5 folds

QT_ROWS=[]

for enc in QT_ENCODINGS:
    for fold in range(5):
        cp=QT_DIR/f"{enc}_fold{fold}.npz"

        if cp.is_file():
            q=np.load(cp,allow_pickle=False)
            m=json.loads(str(q["metrics_json"].item()))
            QT_ROWS.append({"encoding":enc,"fold":fold,**m})
            print(
                "RESUME QT",enc,"fold",fold,
                "acc",round(m["accuracy"],4),
                "auc",round(m["auroc"],4)
            )
            continue

        d=FOLDS[fold]
        print(f"\nQUANTUM TRANSFORMER {enc} fold={fold}")
        t0=time.time()

        model,m,pv,Lv,X8v,ptr,Ltr,X8tr=train_joint_qt(
            d["Z128tr"],d["ytr"],
            d["Z128va"],d["yva"],
            enc,d["bridge_model_path"],
            seed=3000+fold,
            return_train=True
        )
        sec=time.time()-t0

        np.savez_compressed(
            cp,
            train_rows=d["tr"],
            val_rows=d["va"],
            y_train=d["ytr"],
            y_val=d["yva"],
            train_probability=ptr.astype(np.float64),
            val_probability=pv.astype(np.float64),
            latent64_train=Ltr.astype(np.float32),
            latent64_val=Lv.astype(np.float32),
            encoder8_train=X8tr.astype(np.float32),
            encoder8_val=X8v.astype(np.float32),
            metrics_json=np.asarray(json.dumps(m)),
            seconds=np.asarray(sec,np.float64),
        )

        torch.save(
            {
                "state_dict":model.state_dict(),
                "encoding":enc,
                "depth":4,
                "latent_dim":64,
                "joint_encoder":True,
                "attention_equation":"guide-compliant instantiation; source does not specify exact equation",
            },
            QT_DIR/f"{enc}_fold{fold}.pt"
        )

        QT_ROWS.append({"encoding":enc,"fold":fold,**m})
        print("  metrics:",m,"seconds:",round(sec,1))

        del model
        gc.collect()

qt_df=pd.DataFrame(QT_ROWS)
qt_df.to_csv(QT_DIR/"quantum_transformer_fold_metrics.csv",index=False)

qt_summary=rank_summary(qt_df,["encoding"])
qt_summary.to_csv(QT_DIR/"quantum_transformer_summary.csv",index=False)

print("\nQUANTUM TRANSFORMER SUMMARY")
display(qt_summary)

QT_LEADER=str(qt_summary.iloc[0]["encoding"])
print("Standalone Quantum-Transformer leader:",QT_LEADER)


In [ ]:

# Cell 15 — package fold-specific Stage-06 quantum inputs WITHOUT representation leakage

for fold in range(5):
    d=FOLDS[fold]

    va=np.load(VQC_DIR/f"angle_rx_fold{fold}.npz",allow_pickle=False)
    vi=np.load(VQC_DIR/f"iqp_dense_fold{fold}.npz",allow_pickle=False)
    qa=np.load(QT_DIR/f"angle_rx_fold{fold}.npz",allow_pickle=False)
    qi=np.load(QT_DIR/f"iqp_dense_fold{fold}.npz",allow_pickle=False)

    for obj in (va,vi,qa,qi):
        if not np.array_equal(np.asarray(obj["train_rows"],np.int64),d["tr"]):
            raise RuntimeError(f"fold {fold}: train-row mismatch")
        if not np.array_equal(np.asarray(obj["val_rows"],np.int64),d["va"]):
            raise RuntimeError(f"fold {fold}: val-row mismatch")

    np.savez_compressed(
        FOLD_STAGE06_DIR/f"fold{fold}_quantum_inputs.npz",
        train_rows=d["tr"],
        val_rows=d["va"],
        y_train=d["ytr"],
        y_val=d["yva"],

        # Primary proposed-hybrid QML inputs: 8-D quantum measurements.
        vqc_angle8_train=np.asarray(va["q8_train"],np.float32),
        vqc_angle8_val=np.asarray(va["q8_val"],np.float32),
        vqc_iqp8_train=np.asarray(vi["q8_train"],np.float32),
        vqc_iqp8_val=np.asarray(vi["q8_val"],np.float32),

        # Separate guide Quantum-Transformer branch/comparator.
        qt_angle64_train=np.asarray(qa["latent64_train"],np.float32),
        qt_angle64_val=np.asarray(qa["latent64_val"],np.float32),
        qt_iqp64_train=np.asarray(qi["latent64_train"],np.float32),
        qt_iqp64_val=np.asarray(qi["latent64_val"],np.float32),
    )

print("Stage-06 fold-specific quantum packages:",FOLD_STAGE06_DIR)



# D. Final all-learning-record quantum models

After CV evidence is complete, final quantum models are trained on all 35 learning recordings.

These final models are used only to transform:
- all learning rows for the final all-learning Stage-06 fit, and
- official x-record rows for final inference.

For CV evaluation in Stage 06, use the fold-specific files from Cell 15 instead.


In [ ]:

# Cell 16 — final all-learn VQC for both encodings

FINAL_VQC={}

@torch.no_grad()
def infer_final_vqc(model,Z):
    model.eval()
    probs=[]; qfeats=[]
    for st in range(0,len(Z),512):
        xb=torch.tensor(
            Z[st:st+512],dtype=torch.float32,device=QML_DEVICE
        )
        logits,q,_=model(xb)
        probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
        qfeats.append(q.cpu().numpy())
    return np.concatenate(probs),np.concatenate(qfeats).astype(np.float32)

for enc in ENCODINGS:
    print("\nFINAL ALL-LEARN VQC:",enc)

    model,_,_,_,_,_,Qlearn,_=train_joint_vqc(
        FINAL_Z128_LEARN,Y_LEARN,
        FINAL_Z128_LEARN,Y_LEARN,
        enc,FINAL_BRIDGE_MODEL,
        seed=4042,
        return_train=True
    )

    ptest,Qtest=infer_final_vqc(model,FINAL_Z128_TEST)

    torch.save(
        {
            "state_dict":model.state_dict(),
            "encoding":enc,
            "depth":4,
            "joint_encoder":True,
        },
        FINAL_DIR/f"final_vqc_{enc}.pt"
    )

    FINAL_VQC[enc]={"learn":Qlearn,"test":Qtest}
    del model
    gc.collect()

print("Final VQC representations ready:",list(FINAL_VQC))


In [ ]:

# Cell 17 — final all-learn Quantum Transformer for both encodings

FINAL_QT={}

@torch.no_grad()
def infer_final_qt(model,Z):
    model.eval()
    probs=[]; lat=[]
    for st in range(0,len(Z),384):
        xb=torch.tensor(
            Z[st:st+384],dtype=torch.float32,device=QML_DEVICE
        )
        logits,z64,_=model(xb)
        probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
        lat.append(z64.cpu().numpy())
    return np.concatenate(probs),np.concatenate(lat).astype(np.float32)

for enc in QT_ENCODINGS:
    print("\nFINAL ALL-LEARN QUANTUM TRANSFORMER:",enc)

    model,_,_,_,_,_,Llearn,_=train_joint_qt(
        FINAL_Z128_LEARN,Y_LEARN,
        FINAL_Z128_LEARN,Y_LEARN,
        enc,FINAL_BRIDGE_MODEL,
        seed=5042,
        return_train=True
    )

    ptest,Ltest=infer_final_qt(model,FINAL_Z128_TEST)

    torch.save(
        {
            "state_dict":model.state_dict(),
            "encoding":enc,
            "depth":4,
            "latent_dim":64,
            "joint_encoder":True,
        },
        FINAL_DIR/f"final_qt_{enc}.pt"
    )

    FINAL_QT[enc]={"learn":Llearn,"test":Ltest}
    del model
    gc.collect()

np.savez_compressed(
    FINAL_DIR/"quantum_features_for_final_stage06.npz",
    learn_uids=UID_LEARN.astype("U128"),
    test_uids=UID_TEST.astype("U128"),
    y_learn=Y_LEARN,

    # Proposed Stage-06 primary QML features.
    vqc_angle8_learn=FINAL_VQC["angle_rx"]["learn"],
    vqc_angle8_test=FINAL_VQC["angle_rx"]["test"],
    vqc_iqp8_learn=FINAL_VQC["iqp_dense"]["learn"],
    vqc_iqp8_test=FINAL_VQC["iqp_dense"]["test"],

    # Separate Quantum-Transformer branch/comparator.
    qt_angle64_learn=FINAL_QT["angle_rx"]["learn"],
    qt_angle64_test=FINAL_QT["angle_rx"]["test"],
    qt_iqp64_learn=FINAL_QT["iqp_dense"]["learn"],
    qt_iqp64_test=FINAL_QT["iqp_dense"]["test"],
)

print("Final Stage-06 quantum artifact written.")
print("Official test labels used: NO")


In [ ]:

# Cell 18 — final audit report / manifest

BRIDGE_METRICS={
    "accuracy":0.8580743699700405,
    "balanced_accuracy":0.8400134726923901,
    "f1":0.8044358102638821,
    "auroc":0.9209885369613346,
    "auprc":0.8732685926844878,
    "mcc":0.696071416719695,
}

comparison=[{"branch":"Classical bridge comparator","encoding":"S192/drop0.5",**BRIDGE_METRICS}]
for enc in ENCODINGS:
    comparison.append({
        "branch":"Joint VQC",
        "encoding":enc,
        **VQC_OOF[enc]["metrics"]
    })

comparison_df=pd.DataFrame(comparison)
comparison_df.to_csv(OUT/"stage04_vqc_comparison.csv",index=False)
display(comparison_df)

final_stage06_path=FINAL_DIR/"quantum_features_for_final_stage06.npz"

manifest={
    "pipeline":"QML-SleepNet Stage04 AUDITED v3",
    "source_locked":{
        "angle_encoding":"Rx(x_i)",
        "iqp_encoding":"dense all-pairs ZZ",
        "dense_iqp_pair_count":28,
        "classical_encoder":"128->64->32->8",
        "classical_encoder_jointly_trainable_for_vqc_and_qt":True,
        "qubits":8,
        "vqc_depth":4,
        "vqc_quantum_parameters":96,
        "vqc_ansatz":"StronglyEntanglingLayers",
        "entangler":"CNOT",
        "rotation_structure":"Rz-Ry-Rz (contains Ry/Rz)",
        "measurement":"Pauli-Z expectation values",
        "statevector_simulator":True,
        "kernel":"fidelity",
        "kernel_classifier":"classical SVM/QSVM",
        "quantum_transformer_latent_dim":64,
    },
    "implementation_notes":{
        "quantum_transformer_attention_equation_source_status":
            "under-specified by supplied diagram; implemented as labelled guide-compliant instantiation",
        "reduce_lr_on_plateau_source_status":
            "listed in Stage08 but composition with warmup+cosine is not specified; this stage uses warmup+cosine, min_lr=1e-6",
        "loso_source_status":
            "required by Stage08 full training/evaluation protocol; deferred to final full-model evaluation, not used for current five-fold model selection",
    },
    "vqc_summary":vqc_summary.to_dict(orient="records"),
    "standalone_vqc_leader":STANDALONE_VQC_LEADER,
    "kernel_summary":kernel_summary.to_dict(orient="records"),
    "kernel_leader":{"encoding":KERNEL_LEADER_ENCODING,"C":KERNEL_LEADER_C},
    "quantum_transformer_summary":qt_summary.to_dict(orient="records"),
    "quantum_transformer_leader":QT_LEADER,
    "both_vqc_encodings_preserved_for_stage06_hybrid_selection":True,
    "fold_specific_stage06_quantum_features":str(FOLD_STAGE06_DIR),
    "final_stage06_quantum_features":str(final_stage06_path),
    "official_test_labels_used_for_training_or_selection":False,
    "next":"Stage05 causal inference -> Stage06 CNN-BiLSTM -> exact hybrid using fold-safe 8-D VQC features + causal 16-D + classical temporal features",
}

(OUT/"STAGE04_AUDITED_V3_MANIFEST.json").write_text(
    json.dumps(manifest,indent=2)
)

print("="*110)
print("QML-SLEEPNET STAGE04 AUDITED v3 COMPLETE")
print("="*110)
print("Joint VQC leader:",STANDALONE_VQC_LEADER)
print("Kernel leader:",KERNEL_LEADER_ENCODING,"C",KERNEL_LEADER_C)
print("Quantum Transformer leader:",QT_LEADER)
print("Fold-safe Stage06 quantum inputs:",FOLD_STAGE06_DIR)
print("Final Stage06 quantum inputs:",final_stage06_path)
print("Official test labels used: NO")
print("NEXT: exact Stage05 causal module.")
